# STEM Audio Workbench — Free Runner

This notebook runs the STEM Demucs engine directly in Google Colab. It does **not** need Gradio, a share link, a paid API server, Neon credentials, or a browser-side backend URL.

1. Choose **Runtime → Run all**.
2. When the upload cell appears, select your audio file.
3. Wait for separation to finish, listen to the six results, and download the ZIP.

**Important:** Colab sessions are temporary. Keep this tab open until the ZIP downloads. Free GPU availability and session limits are controlled by Google; CPU mode may take much longer. Only upload audio you own or are authorised to process.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install demucs==4.0.1
!ffmpeg -version | head -n 1
!python -m demucs --help > /dev/null && echo 'Demucs installation OK'


In [ ]:
import os, sys, shutil, subprocess, json, uuid
from pathlib import Path
import torch
from IPython.display import Audio, display
from google.colab import files

REPO = Path('/content/STEM')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/demtaiwo1-eng/Stem.git', str(REPO)], check=True)
if not (REPO / 'worker.py').is_file():
    raise RuntimeError('The STEM worker.py file was not found in the repository.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('STEM engine is ready. Device:', DEVICE)
if DEVICE == 'cpu':
    print('No GPU is attached. Processing will use CPU and may be slow. In Colab, try Runtime → Change runtime type → T4 GPU if available.')


In [ ]:
print('Select one audio file to separate (MP3, WAV, FLAC, M4A and other FFmpeg-supported formats).')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No audio file was uploaded. Run this cell again and choose a file.')
filename = next(iter(uploaded))
source_path = Path('/content') / filename
if not source_path.is_file() or source_path.stat().st_size == 0:
    raise RuntimeError('The uploaded file is empty or could not be read.')
job_id = 'stem-' + uuid.uuid4().hex[:12]
job_dir = Path('/content/stem-results') / job_id
job_dir.mkdir(parents=True, exist_ok=True)
print('Uploaded:', source_path.name, '| Size:', round(source_path.stat().st_size / (1024 * 1024), 2), 'MB')
print('Starting six-stem separation. The first run may download the AI model and take several minutes.')
env = os.environ.copy()
env['STEM_DEVICE'] = DEVICE
env['STEM_MODEL'] = 'htdemucs_6s'
command = [sys.executable, str(REPO / 'worker.py'), str(source_path), str(job_dir), job_id]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
log_lines = []
for line in process.stdout:
    line = line.rstrip()
    if line:
        print(line)
        log_lines.append(line)
return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(log_lines[-35:])
    raise RuntimeError('STEM separation failed. Last worker messages:\n' + tail)
output_dir = job_dir / 'deliverables'
manifest_path = output_dir / 'manifest.json'
zip_path = job_dir / 'stem-results.zip'
if not manifest_path.is_file() or not zip_path.is_file():
    raise RuntimeError('The worker finished but did not create the expected manifest and ZIP. Check the worker output above.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('\nCompleted:', len(manifest.get('stems', [])), 'stems | Source duration:', manifest.get('source_duration_seconds'), 'seconds')
print('ZIP ready:', zip_path, '|', round(zip_path.stat().st_size / (1024 * 1024), 2), 'MB')


In [ ]:
stem_files = {item['name']: output_dir / item['mp3'] for item in manifest['stems']}
for stem_name in ['vocals', 'drums', 'bass', 'guitar', 'piano', 'other']:
    stem_path = stem_files.get(stem_name)
    if stem_path and stem_path.is_file():
        print(stem_name.upper())
        display(Audio(filename=str(stem_path)))
print('\nManifest:')
print(json.dumps(manifest, indent=2))
print('\nDownloading the complete ZIP now…')
files.download(str(zip_path))
